# Kotoba TTS v1.0 — Promptless Real-time Text-to-Speech (Bidirectional Streaming)

This notebook demonstrates how to use the Kotoba TTS v1.0 (promptless) model package from AWS Marketplace.

**Product**: Kotoba TTS v1.0 — Promptless Real-time Japanese Speech Synthesis  
**Seller**: Kotoba AI

## Key Features

- **Multiple shipped voice presets** — `default` is always available; additional named presets (e.g. `man-1`, `woman-1`) may also ship with the bundle
- **One-shot synthesis** — submit the full text inside a single `response.create`; the server streams `audio.chunk` deltas back and finishes with `response.done`
- 24 kHz mono `pcm_f32` audio output (little-endian float32)
- Scalable GPU-accelerated inference

## What changed from v0.1

v1.0 is a **different wire protocol** from v0.1, not just a feature bump.

- v0.1 used input-streaming: empty `response.create`, then one or more `input_text_buffer.append` chunks, then `input_text_buffer.commit`. v1.0 is one-shot: the full `text` ships inside `response.create` and `input_text_buffer.*` events no longer exist.
- `response.create` accepts a client-supplied `response_id`. The id is echoed on `response.created`, every `audio.chunk` (as `response_id`), and `response.done` so the client can correlate streams.
- v0.1 shipped only `speaker_id="default"`. v1.0 ships multiple preset voices; clients select among them via `speaker_id`.

The v0.1 protocol reference lives at [docs/tts-v0.1/](../tts-v0.1/).

---

## IMPORTANT: Bidirectional Streaming Only

This product uses **SageMaker bidirectional streaming** exclusively:

- Standard `POST /invocations` is **NOT supported**
- Batch transform jobs are **NOT supported**
- Requires **Python 3.12+** with `aws-sdk-python[sagemaker-runtime-http2]`

---

## Prerequisites

1. **AWS Account** with SageMaker permissions
2. **Python 3.12+** (required for HTTP/2 SDK)
3. **AWS Marketplace subscription** to Kotoba TTS

### Required IAM Permissions

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "sagemaker:CreateEndpoint",
                "sagemaker:CreateEndpointConfig",
                "sagemaker:DeleteEndpoint",
                "sagemaker:DeleteEndpointConfig",
                "sagemaker:DescribeEndpoint",
                "sagemaker:InvokeEndpoint",
                "sagemaker:InvokeEndpointWithResponseStream"
            ],
            "Resource": "*"
        }
    ]
}
```

## Table of Contents

1. [Subscribe to Model Package](#1.-Subscribe-to-Model-Package)
2. [Setup](#2.-Setup)
3. [Deploy Real-time Endpoint](#3.-Deploy-Real-time-Endpoint)
4. [Perform Real-time Synthesis](#4.-Perform-Real-time-Synthesis)
5. [Troubleshooting](#5.-Troubleshooting)
6. [Clean-up](#6.-Clean-up)

## 1. Subscribe to Model Package

To subscribe to the Kotoba TTS model package:

1. Open the [Kotoba TTS listing on AWS Marketplace](#) <!-- TODO: Update with actual URL -->
2. Click **Continue to Subscribe**
3. Review the pricing and EULA, then click **Subscribe**
4. Once subscribed, click **Continue to Configuration**
5. Select your preferred region and note the **Model Package ARN**

## 2. Setup

### 2.1 Install Dependencies

Run this cell first — the Python version check in 2.2 doesn't require these packages, but every cell after that does.

In [ ]:
!pip install -q --upgrade boto3 numpy
!pip install -q --upgrade "sagemaker>=2.0,<3"
!pip install -q --upgrade "aws-sdk-python[sagemaker-runtime-http2]"


### 2.2 Check Python Version

`aws-sdk-python` requires Python 3.12+. If this cell raises, switch your kernel (or create a new environment) before continuing.

In [ ]:
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"Python 3.12+ is required for SageMaker bidirectional streaming. "
        f"Current version: {sys.version}"
    )

print(f"Python version: {sys.version}")


### 2.3 Import Libraries

In [ ]:
import asyncio
import base64
import json
import time
import wave

import boto3
import numpy as np
import sagemaker

# SageMaker HTTP/2 SDK for bidirectional streaming
from aws_sdk_sagemaker_runtime_http2.client import SageMakerRuntimeHTTP2Client
from aws_sdk_sagemaker_runtime_http2.config import Config, HTTPAuthSchemeResolver
from aws_sdk_sagemaker_runtime_http2.models import (
    InvokeEndpointWithBidirectionalStreamInput,
    RequestPayloadPart,
    RequestStreamEventPayloadPart,
)
from sagemaker import ModelPackage
from smithy_aws_core.auth.sigv4 import SigV4AuthScheme
from smithy_aws_core.identity import StaticCredentialsResolver

print("All dependencies imported successfully.")

### 2.4 Configure AWS Session

SageMaker needs an **execution role** (an IAM *role*, not an IAM *user*) to assume when creating the endpoint.

- **On SageMaker Studio / Notebook Instance**: leave `SAGEMAKER_ROLE_ARN = None`. `sagemaker.get_execution_role()` will pick up the attached role automatically.
- **Running locally (IAM user / CLI profile)**: set `SAGEMAKER_ROLE_ARN` to the ARN of a role whose trust policy allows `sagemaker.amazonaws.com` to assume it and which has (at minimum) the `AmazonSageMakerFullAccess` managed policy.


In [ ]:
# --- Execution role -----------------------------------------------------
# If running on SageMaker Studio or a Notebook Instance, leave this as None.
# If running locally as an IAM user, set it to the ARN of a SageMaker-capable role, e.g.
#   SAGEMAKER_ROLE_ARN = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"
SAGEMAKER_ROLE_ARN: str | None = None
# ------------------------------------------------------------------------

sagemaker_session = sagemaker.Session()
region = sagemaker_session.boto_region_name

if SAGEMAKER_ROLE_ARN is not None:
    role = SAGEMAKER_ROLE_ARN
else:
    try:
        role = sagemaker.get_execution_role()
    except ValueError as exc:
        raise ValueError(
            "get_execution_role() only works in SageMaker-managed environments. "
            "Set SAGEMAKER_ROLE_ARN above to a SageMaker execution role ARN "
            "(not your IAM user ARN). See the markdown cell above for role requirements."
        ) from exc

print(f"Region: {region}")
print(f"Role:   {role}")


### 2.5 Configure Model Package ARN

Replace `<MODEL_PACKAGE_ARN>` with the ARN from your AWS Marketplace subscription.

In [ ]:
# TODO: Replace with your Model Package ARN from AWS Marketplace
MODEL_PACKAGE_ARN = "<MODEL_PACKAGE_ARN>"

if MODEL_PACKAGE_ARN.startswith("<"):
    raise ValueError(
        "Please replace MODEL_PACKAGE_ARN with your actual Model Package ARN "
        "from AWS Marketplace subscription."
    )


## 3. Deploy Real-time Endpoint

### 3.1 Create Endpoint

Supported instance types:

| Instance | vCPU | Memory | GPU | Notes |
|----------|------|--------|-----|-------|
| `ml.g6e.8xlarge` | 32 | 256 GiB | 1x NVIDIA L40S | Default, good balance of cost and throughput |
| `ml.g6e.4xlarge` | 16 | 128 GiB | 1x NVIDIA L40S | Choose for cheaper instance                  |


In [ ]:
# Endpoint configuration
ENDPOINT_NAME = f"kotoba-tts-{int(time.time())}"
INSTANCE_TYPE = "ml.g6.2xlarge"  # or "ml.g6.4xlarge"

print(f"Endpoint name: {ENDPOINT_NAME}")
print(f"Instance type: {INSTANCE_TYPE}")


In [ ]:
# Create Model from Model Package
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=sagemaker_session,
)

# Deploy endpoint (this takes 5-10 minutes)
print(f"Deploying endpoint '{ENDPOINT_NAME}'...")

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)

print(f"Endpoint '{ENDPOINT_NAME}' is InService.")


## 4. Perform Real-time Synthesis

### 4.1 Connection Protocol

Kotoba TTS uses SageMaker bidirectional streaming:

```
Client (HTTP/2 SDK) -> SageMaker Runtime (HTTPS:8443) -> Sidecar -> Container (WS:8080)
```

**Note**: Standard WebSocket libraries cannot connect directly to SageMaker endpoints.

### 4.2 Output Audio Format

The server emits a single hardcoded audio format. Audio is base64-encoded
inside `audio.chunk.audio`.

| Format    | Sample Rate | Channels | Encoding              |
| --------- | ----------- | -------- | --------------------- |
| `pcm_f32` | 24000 Hz    | 1 (mono) | Little-endian float32 |

### 4.3 Protocol Flow

The client must send `open` first; the server replies with `session.created`
once the language and speaker are validated. If `open` is not sent within
`WS_OPEN_TIMEOUT` (default 10s), the server closes the connection.

```
Client                                    Server
  |---- open ------------------------------>|
  |     (language, speaker_id)              |
  |<---- session.created -------------------|
  |---- response.create ------------------->|
  |     (text, response_id?)                |
  |<---- response.created ------------------|
  |     (response.id)                       |
  |<---- audio.chunk (isFinal:false) -------|
  |     (response_id, base64 pcm_f32)       |
  |              ...                        |
  |<---- audio.chunk (isFinal:true) --------|
  |<---- response.done ---------------------|
  |     (response.id, status: completed)    |
```

### 4.4 Event Reference

#### Client Events (you send)

| Event              | Description                                                                                          |
| ------------------ | ---------------------------------------------------------------------------------------------------- |
| `open`             | First message; configure `language`, `speaker_id`                                                    |
| `response.create`  | Submit the full text for synthesis. Required: `text`. Optional: `response_id`. Repeatable per session. |
| `response.cancel`  | Cancel the currently active synthesis                                                                |

> v1.0 has **no `input_text_buffer.append` or `input_text_buffer.commit`** —
> the full text ships in `response.create`. The v0.1 input-streaming protocol
> is documented at [docs/tts-v0.1/](../tts-v0.1/).

#### Server Events (you receive)

| Event              | Description                                                                                                  |
| ------------------ | ------------------------------------------------------------------------------------------------------------ |
| `session.created`  | Session established (`format`, `sample_rate`, `channels`, `language`, `speaker_id`, `client_id`)             |
| `response.created` | Synthesis turn started. Payload: `response: {id}` (the id supplied in `response.create`, or a server UUID).  |
| `audio.chunk`      | Base64 audio delta. Payload: `response_id`, `audio` (base64 pcm_f32), `isFinal` (true marks the last chunk). |
| `response.done`    | Turn finished. Payload: `response: {id, status, error?, usage?}` where `status` is `completed`/`cancelled`/`failed`. `usage` = `{input_tokens, output_tokens, total_tokens}`, present when tokens were consumed. |
| `error`            | Fatal error (connection will close)                                                                          |
| `timeout`          | Worker result timeout notification (session still alive)                                                     |

See [docs/tts/data/sample_input.json](./data/sample_input.json) and [docs/tts/data/sample_output.json](./data/sample_output.json) for canonical event payloads.

### 4.5 Prepare Input

`SPEAKER_ID` selects one of the available voice presets. `default` is always
available; additional preset keys depend on the bundle.

v1.0 is one-shot: assemble the full text and send it inside a single
`response.create`. Sentence-level chunking is not required.

In [ ]:
SAMPLE_RATE = 24000
LANGUAGE = "ja"

# Select an available preset voice. "default" is always shipped. Other preset
# keys (e.g. "man-1", "woman-1") depend on the bundle.
SPEAKER_ID = "default"

# Full text to synthesize. v1.0 is one-shot: the entire string is sent inside
# a single response.create event.
TEXT = "こんにちは、世界。今日もよろしくお願いします。"

# Optional. When provided, echoed back on response.created / audio.chunk /
# response.done so the client can correlate streams. Omit to let the server
# assign a UUID.
RESPONSE_ID = "resp_001"

print(f"Language:    {LANGUAGE}")
print(f"Speaker:     {SPEAKER_ID}")
print(f"Response id: {RESPONSE_ID}")
print(f"Text:        {TEXT!r} ({len(TEXT)} chars)")


### 4.6 Streaming Inference

In [ ]:
async def synthesize_text(
    endpoint_name: str,
    text: str,
    language: str = "ja",
    speaker_id: str = "default",
    response_id: str | None = None,
) -> bytes:
    """
    Synthesize speech using Kotoba TTS v1.0 (promptless, one-shot) bidirectional streaming.

    Args:
        endpoint_name: SageMaker endpoint name.
        text: full text to synthesize. Sent inside a single response.create
            event; v1.0 does not stream text chunks.
        language: target language (must be in the endpoint's
            `SUPPORTED_LANGUAGES`).
        speaker_id: preset voice identifier. `default` is always available;
            additional preset keys depend on the bundle.
        response_id: optional client-supplied response id. Echoed on
            response.created / audio.chunk / response.done. When omitted the
            server assigns a UUID.

    Returns:
        Raw audio bytes (pcm_f32, 24000 Hz, mono).
    """
    # Get AWS credentials
    session = boto3.Session()
    credentials = session.get_credentials().get_frozen_credentials()

    # Configure HTTP/2 client
    config = Config(
        endpoint_uri=f"https://runtime.sagemaker.{region}.amazonaws.com:8443",
        region=region,
        aws_access_key_id=credentials.access_key,
        aws_secret_access_key=credentials.secret_key,
        aws_session_token=credentials.token,
        aws_credentials_identity_resolver=StaticCredentialsResolver(),
        auth_scheme_resolver=HTTPAuthSchemeResolver(),
        auth_schemes={"aws.auth#sigv4": SigV4AuthScheme(service="sagemaker")},
    )

    client = SageMakerRuntimeHTTP2Client(config=config)

    # Start bidirectional stream
    stream = await client.invoke_endpoint_with_bidirectional_stream(
        InvokeEndpointWithBidirectionalStreamInput(
            endpoint_name=endpoint_name,
            model_invocation_path="invocations-bidirectional-stream",
        )
    )

    async def send_event(payload: dict):
        body = json.dumps(payload).encode("utf-8")
        # NOTE: data_type="UTF8" is required so SageMaker forwards the frame
        # as text. Without it the server (which calls ws.receive_text()) cannot
        # decode the payload and the connection fails before any handler runs.
        request = RequestStreamEventPayloadPart(
            value=RequestPayloadPart(bytes_=body, data_type="UTF8")
        )
        await stream.input_stream.send(request)

    async def recv_event() -> dict:
        message = await output_stream.receive()
        return json.loads(message.value.bytes_.decode())

    output = await stream.await_output()
    output_stream = output[1] if isinstance(output, tuple) else output

    # 1) Send open. The server waits for this within WS_OPEN_TIMEOUT (~10s)
    #    and replies with session.created once language/speaker are validated.
    await send_event({
        "type": "open",
        "language": language,
        "speaker_id": speaker_id,
    })

    # 2) Receive session.created
    data = await recv_event()
    assert data["type"] == "session.created", data
    print(
        f"Session created: format={data.get('format')} sr={data.get('sample_rate')} "
        f"ch={data.get('channels')} speaker_id={data.get('speaker_id')}"
    )

    # 3) Submit the full text in a single response.create
    create_payload: dict = {"type": "response.create", "text": text}
    if response_id is not None:
        create_payload["response_id"] = response_id
    await send_event(create_payload)

    # 4) Receive response.created and capture the server-assigned id (if the
    #    client did not supply one).
    data = await recv_event()
    assert data["type"] == "response.created", data
    server_response_id = (data.get("response") or {}).get("id")
    print(f"Response started: id={server_response_id}")

    # 5) Collect audio.chunk deltas until response.done.
    audio_buffer = bytearray()
    while True:
        data = await recv_event()
        event_type = data["type"]
        if event_type == "audio.chunk":
            audio_b64 = data.get("audio", "")
            is_final = data.get("isFinal", False)
            audio_buffer.extend(base64.b64decode(audio_b64))
            print(
                f"Got audio chunk: response_id={data.get('response_id')} "
                f"b64_chars={len(audio_b64)} isFinal={is_final}"
            )
            continue
        if event_type == "response.done":
            response = data.get("response", {})
            status = response.get("status")
            print(f"Response done: id={response.get('id')} status={status} usage={response.get('usage')}")
            if status != "completed":
                raise RuntimeError(f"Synthesis failed: {data}")
            break
        if event_type in ("error", "timeout"):
            # `timeout` is non-fatal on its own; after WS_RESULT_TIMEOUT_LIMIT
            # consecutive timeouts the server emits a fatal `error` and closes.
            raise RuntimeError(f"Server event: {data}")

    await stream.input_stream.close()
    return bytes(audio_buffer)


In [ ]:
# Run synthesis
print("Starting synthesis...\n")

audio_bytes = await synthesize_text(
    endpoint_name=ENDPOINT_NAME,
    text=TEXT,
    language=LANGUAGE,
    speaker_id=SPEAKER_ID,
    response_id=RESPONSE_ID,
)

print(f"\nReceived {len(audio_bytes)} bytes of pcm_f32 audio")

# Convert pcm_f32 -> int16 and save as WAV for easy playback
samples_f32 = np.frombuffer(audio_bytes, dtype=np.float32)
samples_i16 = np.clip(samples_f32, -1.0, 1.0) * 32767.0
samples_i16 = samples_i16.astype(np.int16)

out_path = "sample_output.wav"
with wave.open(out_path, "wb") as w:
    w.setnchannels(1)
    w.setsampwidth(2)
    w.setframerate(SAMPLE_RATE)
    w.writeframes(samples_i16.tobytes())

print(f"Saved synthesized audio to {out_path} ({len(samples_i16) / SAMPLE_RATE:.2f} s)")


## 5. Troubleshooting

| Symptom | Likely cause & fix |
|---------|--------------------|
| `RuntimeError: Python 3.12+ is required` | The HTTP/2 SDK requires Python 3.12+. Switch your kernel or create a new conda/venv with Python 3.12. |
| `ModuleNotFoundError: aws_sdk_sagemaker_runtime_http2` | Re-run the install cell in 2.1 (`pip install aws-sdk-python[sagemaker-runtime-http2]`). The package name uses underscores when imported. |
| Connection closes immediately after invoke (no `session.created`) | The first event must be `open` and must be sent within `WS_OPEN_TIMEOUT` (default 10s). Make sure `RequestPayloadPart` is constructed with `data_type="UTF8"` so SageMaker forwards a text frame. |
| Server replies with `error` / `unsupported_language` | The endpoint's `SUPPORTED_LANGUAGES` does not include the requested language. Use `language="ja"`. |
| Server replies with `error` / `invalid message type` after sending `input_text_buffer.append` | You are using the v0.1 input-streaming protocol against a v1.0 endpoint. v1.0 is one-shot: put the full text inside `response.create` instead. See [docs/tts-v0.1/](../tts-v0.1/) for the v0.1 protocol. |
| `response.done` with `status: failed` and `unknown_speaker_id` | The requested `speaker_id` is not one of the available presets. Use `"default"` (always available) or another preset key shipped with the bundle. |
| `error` / `text must be a non-empty string` | `response.create` requires a non-empty `text` field. v1.0 does not accept an empty `response.create` followed by `input_text_buffer.append`. |
| `error` / `response already active` | Wait for `response.done` before sending the next `response.create`, or send `response.cancel` first. Only one response can be active per session. |
| `timeout` event during synthesis | The server did not produce audio within `WS_RESULT_TIMEOUT` (default 60s). Usually transient — retry the response. After `WS_RESULT_TIMEOUT_LIMIT` (default 3) consecutive timeouts the server escalates to a fatal `error` and closes. |
| Idle connection closed after ~60s | Hit `WS_IDLE_TIMEOUT`. Send a new `response.create` (or any client event) before the timeout, or open a fresh connection per turn. |
| `AccessDeniedException` at `model.deploy(...)` | The execution role cannot pull the Marketplace model. Verify the AWS Marketplace subscription is active and that the role has `AmazonSageMakerFullAccess` (or equivalent). |
| Saved WAV is silent / sounds like white noise | The pcm_f32 -> int16 conversion was skipped or the byte order is wrong. The server emits little-endian float32; decode with `np.frombuffer(..., dtype=np.float32)` before scaling. |

## 6. Clean-up

**Important**: Delete the endpoint when done. GPU instances (`ml.g6e.8xlarge` / `ml.g6e.4xlarge`) are billed **per hour while the endpoint is active**, even when idle.

In [ ]:
print(f"Deleting endpoint '{ENDPOINT_NAME}'...")

sagemaker_session.delete_endpoint(ENDPOINT_NAME)
sagemaker_session.delete_endpoint_config(ENDPOINT_NAME)

print("Endpoint deleted.")


### Unsubscribe from Model Package (Optional)

To unsubscribe from the Kotoba TTS model package:

1. Go to [AWS Marketplace Subscriptions](https://console.aws.amazon.com/marketplace/home#/subscriptions)
2. Find "Kotoba TTS" in your subscriptions
3. Click **Manage** -> **Actions** -> **Cancel subscription**

**Note**: You must delete all endpoints using this model package before unsubscribing.

---

## Support

For technical support, please contact Kotoba AI support.

## References

- Canonical protocol samples: [docs/tts/data/sample_input.json](./data/sample_input.json), [docs/tts/data/sample_output.json](./data/sample_output.json), [docs/tts/data/README.md](./data/README.md)
- Previous version docs (input-streaming protocol): [docs/tts-v0.1/](../tts-v0.1/)
- [SageMaker Bidirectional Streaming Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-test-endpoints.html)
- [AWS SDK for Python (Boto3)](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)